# MLP for Classification: MNIST / sklearn Digits

---

## Learning Objectives

By the end of this notebook, you will:

- Load and preprocess image classification data (MNIST or sklearn digits)
- Build an MLP classifier with `nn.Module`
- Train with `CrossEntropyLoss` and evaluate accuracy
- Analyze per-class performance with a confusion matrix
- Build a complete end-to-end classification pipeline

## Prerequisites

- **DL100**: Neural network fundamentals
- **DL150**: PyTorch foundations
- **Notebook 01**: MLP for regression (training loops, `nn.Module`)

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [Load the Data](#2.-Load-the-Data)
3. [Explore and Visualize](#3.-Explore-and-Visualize)
4. [Prepare Data for the MLP](#4.-Prepare-Data-for-the-MLP)
5. [Build the MLP Classifier](#5.-Build-the-MLP-Classifier)
6. [Training Loop](#6.-Training-Loop)
7. [Evaluation: Accuracy and Confusion Matrix](#7.-Evaluation:-Accuracy-and-Confusion-Matrix)
8. [Per-Class Performance](#8.-Per-Class-Performance)
9. [Common Mistakes and Debugging Tips](#9.-Common-Mistakes-and-Debugging-Tips)
10. [Exercises](#10.-Exercises)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed
from src.utils.device import get_device
from src.utils.plotting import plot_loss_curves, plot_confusion_matrix
from src.utils.metrics import compute_accuracy, compute_classification_report

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

set_global_seed(42)
device = get_device()

print("Setup complete.")

---

## 2. Load the Data

We try to load MNIST from torchvision first (28x28 images, 70k samples). If that fails
(e.g., no internet, download issues), we fall back to `sklearn.datasets.load_digits`
(8x8 images, 1797 samples).

In [ ]:
USE_MNIST = False  # will be set to True if MNIST loads successfully

try:
    from torchvision import datasets, transforms
    import os

    data_dir = os.path.join("..", "..", "data")
    mnist_train = datasets.MNIST(data_dir, train=True, download=True)
    mnist_test = datasets.MNIST(data_dir, train=False, download=True)

    X_train_raw = mnist_train.data.numpy().astype(np.float32)
    y_train_raw = mnist_train.targets.numpy()
    X_test_raw = mnist_test.data.numpy().astype(np.float32)
    y_test_raw = mnist_test.targets.numpy()

    img_height, img_width = 28, 28
    USE_MNIST = True
    print(f"Loaded MNIST: {X_train_raw.shape[0]} train, {X_test_raw.shape[0]} test")
    print(f"Image size: {img_height}x{img_width}")

except Exception as e:
    print(f"MNIST download failed: {e}")
    print("Falling back to sklearn digits dataset.\n")

    from sklearn.datasets import load_digits

    digits = load_digits()
    X_all = digits.data.astype(np.float32)  # already flattened (1797, 64)
    y_all = digits.target

    img_height, img_width = 8, 8

    # Split into train/test
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X_all.reshape(-1, img_height, img_width), y_all,
        test_size=0.2, random_state=42, stratify=y_all
    )
    print(f"Loaded sklearn digits: {X_train_raw.shape[0]} train, {X_test_raw.shape[0]} test")
    print(f"Image size: {img_height}x{img_width}")

n_classes = 10
input_dim = img_height * img_width
print(f"\nClasses: {n_classes}, Input dim (flattened): {input_dim}")

---

## 3. Explore and Visualize

In [ ]:
# Show sample images
fig, axes = plt.subplots(2, 10, figsize=(14, 3))

for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train_raw[i], cmap="gray")
    ax.set_title(str(y_train_raw[i]), fontsize=9)
    ax.axis("off")

plt.suptitle("Sample Training Images", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (data, title) in zip(axes, [(y_train_raw, "Train"), (y_test_raw, "Test")]):
    unique, counts = np.unique(data, return_counts=True)
    ax.bar(unique, counts, color="steelblue", edgecolor="black", alpha=0.8)
    ax.set_xlabel("Digit")
    ax.set_ylabel("Count")
    ax.set_title(f"{title} Class Distribution")
    ax.set_xticks(range(10))
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Pixel intensity statistics
print(f"Pixel value range: [{X_train_raw.min():.1f}, {X_train_raw.max():.1f}]")
print(f"Mean pixel value:  {X_train_raw.mean():.2f}")
print(f"Std pixel value:   {X_train_raw.std():.2f}")

---

## 4. Prepare Data for the MLP

An MLP takes **flat vectors** as input. We flatten each image from `(H, W)` to `(H*W,)`
and normalize pixel values.

$$\mathbf{x}_{\text{flat}} = \text{reshape}(\mathbf{X}_{H \times W}) \in \mathbb{R}^{H \cdot W}$$

In [ ]:
# Flatten images
X_train_flat = X_train_raw.reshape(X_train_raw.shape[0], -1)  # (N, H*W)
X_test_flat = X_test_raw.reshape(X_test_raw.shape[0], -1)

print(f"Flattened shapes: train={X_train_flat.shape}, test={X_test_flat.shape}")

# Normalize: scale to [0, 1] by dividing by max pixel value
max_pixel = X_train_flat.max()
X_train_norm = X_train_flat / max_pixel
X_test_norm = X_test_flat / max_pixel

print(f"After normalization: [{X_train_norm.min():.2f}, {X_train_norm.max():.2f}]")

In [ ]:
# Create a validation split from training data
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_norm, y_train_raw, test_size=0.15, random_state=42, stratify=y_train_raw
)

print(f"Train:      {X_train_final.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test:       {X_test_norm.shape[0]} samples")

In [ ]:
# Convert to PyTorch tensors and DataLoaders
def make_classification_loaders(X_tr, y_tr, X_val, y_val, X_te, y_te, batch_size=128):
    """Create train, val, and test DataLoaders."""
    train_ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.long),
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long),
    )
    test_ds = TensorDataset(
        torch.tensor(X_te, dtype=torch.float32),
        torch.tensor(y_te, dtype=torch.long),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    return train_loader, val_loader, test_loader


BATCH_SIZE = 128
train_loader, val_loader, test_loader = make_classification_loaders(
    X_train_final, y_train_final, X_val, y_val, X_test_norm, y_test_raw,
    batch_size=BATCH_SIZE
)
print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

---

## 5. Build the MLP Classifier

Architecture:

$$\text{Input}(d) \to \text{Linear}(256) \to \text{ReLU} \to \text{Linear}(128) \to \text{ReLU} \to \text{Linear}(10)$$

**Important:** The output layer has **no softmax**. PyTorch's `CrossEntropyLoss` expects raw logits
and applies `LogSoftmax + NLLLoss` internally.

$$\mathcal{L}_{\text{CE}} = -\frac{1}{N}\sum_{i=1}^{N} \log\left(\frac{e^{z_{y_i}}}{\sum_{j=1}^{C} e^{z_j}}\right)$$

where $z_j$ are the logits and $y_i$ is the true class.

In [ ]:
class ClassifierMLP(nn.Module):
    """Multi-layer perceptron for multi-class classification."""

    def __init__(self, input_dim, n_classes, hidden_dims=(256, 128), dropout=0.0):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, n_classes))  # raw logits
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)  # returns logits (N, n_classes)


set_global_seed(42)
model = ClassifierMLP(input_dim=input_dim, n_classes=n_classes, hidden_dims=(256, 128)).to(device)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}")

---

## 6. Training Loop

In [ ]:
def train_classifier(model, train_loader, val_loader, epochs=30, lr=1e-3, device=device):
    """Train a classifier. Returns training history."""
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc": [], "val_acc": [],
    }

    for epoch in range(1, epochs + 1):
        # -- Training --
        model.train()
        train_losses = []
        correct, total = 0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            preds = logits.argmax(dim=-1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

        avg_train_loss = np.mean(train_losses)
        train_acc = correct / total
        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(train_acc)

        # -- Validation --
        model.eval()
        val_losses = []
        correct, total = 0, 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                logits = model(X_batch)
                val_losses.append(loss_fn(logits, y_batch).item())
                preds = logits.argmax(dim=-1)
                correct += (preds == y_batch).sum().item()
                total += y_batch.size(0)

        avg_val_loss = np.mean(val_losses)
        val_acc = correct / total
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(val_acc)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

    return history

In [ ]:
set_global_seed(42)
model = ClassifierMLP(input_dim=input_dim, n_classes=n_classes, hidden_dims=(256, 128)).to(device)

EPOCHS = 30
LR = 1e-3

history = train_classifier(model, train_loader, val_loader, epochs=EPOCHS, lr=LR)

In [ ]:
# Plot training curves: loss and accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, EPOCHS + 1)

# Loss
ax1.plot(epochs_range, history["train_loss"], label="Train Loss", marker="o", markersize=3)
ax1.plot(epochs_range, history["val_loss"], label="Val Loss", marker="s", markersize=3)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Cross-Entropy Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs_range, history["train_acc"], label="Train Acc", marker="o", markersize=3)
ax2.plot(epochs_range, history["val_acc"], label="Val Acc", marker="s", markersize=3)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Classification Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 7. Evaluation: Accuracy and Confusion Matrix

In [ ]:
@torch.no_grad()
def get_all_predictions(model, loader, device=device):
    """Get all predictions and true labels from a DataLoader."""
    model.eval()
    all_preds = []
    all_labels = []
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(y_batch.numpy())
    return np.concatenate(all_preds), np.concatenate(all_labels)


# Test set evaluation
y_pred_test, y_true_test = get_all_predictions(model, test_loader)
test_acc = (y_pred_test == y_true_test).mean()
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

In [ ]:
# Confusion matrix
class_names = [str(i) for i in range(10)]
plot_confusion_matrix(y_true_test, y_pred_test, class_names=class_names,
                      title="Test Set Confusion Matrix")

In [ ]:
# Show some misclassified examples
misclassified = np.where(y_pred_test != y_true_test)[0]
n_show = min(20, len(misclassified))

if n_show > 0:
    fig, axes = plt.subplots(2, min(10, n_show), figsize=(14, 3))
    axes = axes.flatten() if n_show > 1 else [axes]

    for i, idx in enumerate(misclassified[:n_show]):
        img = X_test_raw[idx]
        axes[i].imshow(img, cmap="gray")
        axes[i].set_title(f"T:{y_true_test[idx]} P:{y_pred_test[idx]}", fontsize=8,
                         color="red")
        axes[i].axis("off")

    # Hide unused axes
    for i in range(n_show, len(axes)):
        axes[i].axis("off")

    plt.suptitle("Misclassified Examples (T=True, P=Predicted)", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified examples! Perfect accuracy.")

print(f"\n{len(misclassified)} misclassified out of {len(y_true_test)} test samples")

---

## 8. Per-Class Performance

In [ ]:
# Detailed classification report
report = compute_classification_report(y_true_test, y_pred_test,
                                        target_names=class_names)

# Display as a formatted table
print(f"{'Class':<10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print("-" * 52)
for cls_name in class_names:
    r = report[cls_name]
    print(f"{cls_name:<10} {r['precision']:>10.4f} {r['recall']:>10.4f} "
          f"{r['f1-score']:>10.4f} {r['support']:>10.0f}")

print("-" * 52)
acc = report["accuracy"]
print(f"{'Accuracy':<10} {'':>10} {'':>10} {acc:>10.4f} {sum(r['support'] for r in [report[c] for c in class_names]):>10.0f}")

In [ ]:
# Bar chart of per-class F1 scores
f1_scores = [report[c]["f1-score"] for c in class_names]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["green" if f1 > 0.95 else "orange" if f1 > 0.90 else "red" for f1 in f1_scores]
ax.bar(class_names, f1_scores, color=colors, edgecolor="black", alpha=0.8)
ax.set_xlabel("Digit")
ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1 Scores")
ax.set_ylim(0.8, 1.01)
ax.axhline(0.95, color="gray", linestyle="--", alpha=0.5, label="0.95 threshold")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

worst_class = class_names[np.argmin(f1_scores)]
best_class = class_names[np.argmax(f1_scores)]
print(f"Best class:  {best_class} (F1={max(f1_scores):.4f})")
print(f"Worst class: {worst_class} (F1={min(f1_scores):.4f})")

---

## 9. Common Mistakes and Debugging Tips

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Applying softmax before `CrossEntropyLoss` | Lower accuracy, numerically unstable | Pass raw logits to `CrossEntropyLoss` |
| Using `float` targets with `CrossEntropyLoss` | Runtime error or silent wrong results | Use `torch.long` (integer) labels |
| Not flattening images | Shape error in first linear layer | Reshape `(N, H, W)` to `(N, H*W)` |
| Forgetting `model.eval()` during evaluation | Dropout still active, inconsistent results | Toggle `train()` / `eval()` |
| Wrong output dimension | Loss has wrong shape | Output dim must equal number of classes |
| Not shuffling training data | Model memorizes order, poor generalization | Set `shuffle=True` in train DataLoader |
| Using test set for validation | Overly optimistic estimates | Split train into train+val, keep test separate |

---

## 10. Exercises

### Exercise 1: Hidden Size Experiment

Train classifiers with different hidden sizes and compare:

- Small: `(64, 32)`
- Medium: `(256, 128)` (our default)
- Large: `(512, 256, 128)`

Plot validation accuracy over epochs for each.

In [ ]:
# ===== EXERCISE 1: Your code here =====
# configs = {
#     "Small (64,32)": (64, 32),
#     "Medium (256,128)": (256, 128),
#     "Large (512,256,128)": (512, 256, 128),
# }
#
# for name, hidden_dims in configs.items():
#     set_global_seed(42)
#     m = ClassifierMLP(input_dim, n_classes, hidden_dims=hidden_dims).to(device)
#     h = train_classifier(m, train_loader, val_loader, epochs=20, lr=1e-3)
#     print(f"{name}: Final Val Acc = {h['val_acc'][-1]:.4f}")

### Exercise 2: Activation Functions

Modify `ClassifierMLP` to accept different activation functions. Compare ReLU, Tanh, and LeakyReLU.

In [ ]:
# ===== EXERCISE 2: Your code here =====
# Hint: Add an `activation` parameter to the constructor
# activation_map = {"relu": nn.ReLU(), "tanh": nn.Tanh(), "leaky_relu": nn.LeakyReLU()}
pass

### Exercise 1 -- Solution

In [ ]:
configs = {
    "Small (64,32)": (64, 32),
    "Medium (256,128)": (256, 128),
    "Large (512,256,128)": (512, 256, 128),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
all_histories = {}

for name, hidden_dims in configs.items():
    set_global_seed(42)
    m = ClassifierMLP(input_dim, n_classes, hidden_dims=hidden_dims).to(device)
    n_p = sum(p.numel() for p in m.parameters())
    h = train_classifier(m, train_loader, val_loader, epochs=20, lr=1e-3)
    all_histories[name] = h

    ep = range(1, 21)
    ax1.plot(ep, h["val_loss"], label=f"{name} ({n_p:,} params)")
    ax2.plot(ep, h["val_acc"], label=f"{name} ({n_p:,} params)")
    print(f"{name:25s} | Params: {n_p:>8,} | Final Val Acc: {h['val_acc'][-1]:.4f}")

ax1.set_xlabel("Epoch"); ax1.set_ylabel("Val Loss"); ax1.set_title("Validation Loss")
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Val Acc"); ax2.set_title("Validation Accuracy")
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise 2 -- Solution

In [ ]:
class FlexibleMLP(nn.Module):
    """MLP with configurable activation function."""

    def __init__(self, input_dim, n_classes, hidden_dims=(256, 128), activation="relu"):
        super().__init__()
        act_map = {
            "relu": nn.ReLU,
            "tanh": nn.Tanh,
            "leaky_relu": nn.LeakyReLU,
        }
        act_cls = act_map.get(activation, nn.ReLU)

        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h_dim), act_cls()])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, n_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


activations = ["relu", "tanh", "leaky_relu"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for act_name in activations:
    set_global_seed(42)
    m = FlexibleMLP(input_dim, n_classes, hidden_dims=(256, 128), activation=act_name).to(device)
    h = train_classifier(m, train_loader, val_loader, epochs=20, lr=1e-3)

    ep = range(1, 21)
    ax1.plot(ep, h["val_loss"], label=act_name)
    ax2.plot(ep, h["val_acc"], label=act_name)
    print(f"{act_name:12s} | Final Val Acc: {h['val_acc'][-1]:.4f}")

ax1.set_xlabel("Epoch"); ax1.set_ylabel("Val Loss"); ax1.set_title("Val Loss by Activation")
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Val Acc"); ax2.set_title("Val Accuracy by Activation")
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nReLU typically converges fastest for deep networks.")
print("Tanh can work well but may suffer from vanishing gradients in deeper nets.")
print("LeakyReLU avoids dead neurons (zero gradient for negative inputs).")

---

**Next notebook:** [03 -- Hyperparameter Tuning and Learning Curves](03_Hyperparameter_Tuning_and_Learning_Curves.ipynb)